# Routing Backends

`idd-travel-time` uses an abstract `Router` interface so the backend is a
one-line swap. This notebook explains what each backend does, when to use it,
and how to implement a custom one.

**What's covered**
1. `OsmnxRouter` — the local default (no key, Overpass download)
2. `OpenRouteServiceRouter` — the cloud free-tier backend (ORS key needed)
3. `GoogleMapsRouter` — paid stub (interface only, not implemented)
4. The `Router` abstract interface
5. Implementing a custom backend
6. Auto-selection logic in `compute()`
7. Choosing the right backend for your use case


## 1 · OsmnxRouter — local, no API key

In [ ]:
from idd_travel_time.constants import TransportMode
from idd_travel_time.isochrone import compute
from idd_travel_time.models import TravelTimeRequest
from idd_travel_time.routers import OsmnxRouter

router = OsmnxRouter()
print(f"Name:         {router.name}")
print(f"Graph cache:  {router._cache}")

request = TravelTimeRequest(
    origin_lat=-1.2864,
    origin_lon=36.8172,
    mode=TransportMode.WALKING,
    duration_minutes=20,
)
result = compute(request, router=router)
print(f"\nRouter used: {result.router_name}")

### OsmnxRouter with graph cache

Pass a `GraphCache` to avoid re-downloading the street graph on every run.


In [ ]:
import tempfile
from pathlib import Path

from idd_travel_time.io import GraphCache

cache = GraphCache(Path(tempfile.mkdtemp()) / "graphs")
cached_router = OsmnxRouter(graph_cache=cache)

# First run downloads; second run is instant
result = compute(request, router=cached_router)
result = compute(request, router=cached_router)
print(f"Cache contains {len(list(cache.cache_dir.glob('*.graphml')))} graph(s)")

### network_type parameter

Controls which OSM edges are included. `"all"` (default) covers roads,
paths, and cycleways. Use `"walk"` or `"drive"` to restrict.


In [ ]:
walk_router = OsmnxRouter(network_type="walk")  # pedestrian paths only
drive_router = OsmnxRouter(network_type="drive")  # roads only

result_walk = compute(request, router=walk_router)
result_drive = compute(
    TravelTimeRequest(
        origin_lat=-1.2864,
        origin_lon=36.8172,
        mode=TransportMode.DRIVING,
        duration_minutes=10,
    ),
    router=drive_router,
)


def area_km2(r):
    p = r.isochrone_gdf.to_crs(r.isochrone_gdf.estimate_utm_crs())
    return p.geometry.area.iloc[0] / 1e6


print(f"Walking network:   {area_km2(result_walk):.2f} km²")
print(f"Driving network:   {area_km2(result_drive):.2f} km²")

## 2 · OpenRouteServiceRouter — cloud, free tier

In [ ]:
import os

from idd_travel_time.routers import OpenRouteServiceRouter

if os.environ.get("ORS_API_KEY"):
    ors_router = OpenRouteServiceRouter()
    print(f"Name: {ors_router.name}")

    ors_result = compute(request, router=ors_router)
    print(f"Router used: {ors_result.router_name}")
    print(f"Area: {area_km2(ors_result):.2f} km²")
else:
    print("ORS_API_KEY not set.")
    print("Get a free key at https://openrouteservice.org/dev/#/signup")
    print("Then: export ORS_API_KEY=your_key_here")
    print()
    print("The ORS router is otherwise a one-line swap:")
    print("  router = OpenRouteServiceRouter()")
    print("  result = compute(request, router=router)")

### Transit scaling in ORS

ORS has no transit profile. For `TRANSIT` mode, `OpenRouteServiceRouter`
uses the driving-car network and scales the time budget by the ratio of
the `TransitProfile` midpoint speed to ORS's implicit 30 kph urban speed.

This means a `TransitProfile` midpoint of 15 kph produces a range half
as large as a pure 30-min driving isochrone — correct.


In [ ]:
from idd_travel_time.models import TransitProfile
from idd_travel_time.routers.openroute import (
    _ORS_DRIVING_SPEED_KPH,
    OpenRouteServiceRouter,
)

profile = TransitProfile.from_preset("informal_minibus")
scale = profile.midpoint_speed_kph / _ORS_DRIVING_SPEED_KPH
print(f"informal_minibus midpoint: {profile.midpoint_speed_kph:.1f} kph")
print(f"ORS driving assumption:    {_ORS_DRIVING_SPEED_KPH:.0f} kph")
print(f"Range scale factor:        {scale:.2f}×")
print(f"Effective 30-min budget:   {30 * 60 * scale:.0f}s")

## 3 · GoogleMapsRouter — paid stub

In [ ]:
import os

from idd_travel_time.routers import GoogleMapsRouter

# Requires GOOGLE_MAPS_API_KEY — currently a stub (NotImplementedError)
if os.environ.get("GOOGLE_MAPS_API_KEY"):
    gmaps = GoogleMapsRouter()
    print(f"Name: {gmaps.name}")
    try:
        gmaps.compute_isochrone(request)
    except NotImplementedError as e:
        print(f"Not yet implemented: {e}")
else:
    print("GoogleMapsRouter is a paid stub.")
    print("Interface is fully wired — swap in with one line when a key is available:")
    print()
    print("  router = GoogleMapsRouter()           # or pass api_key= explicitly")
    print("  result = compute(request, router=router)")

## 4 · The Router abstract interface

In [ ]:
import inspect

from idd_travel_time.routers.base import Router

print(inspect.getsource(Router))

## 5 · Implementing a custom backend

Any class that inherits from `Router` and implements `name` and
`compute_isochrone()` is a valid backend. Here's a minimal example
that wraps a hypothetical travel-time matrix API.


In [ ]:
from shapely.geometry import Point

from idd_travel_time.models import TravelTimeRequest, TravelTimeResult
from idd_travel_time.routers.base import Router


class MyCustomRouter(Router):
    """Minimal example — replaces ORS/osmnx with any backend."""

    def __init__(self, api_endpoint: str) -> None:
        self._endpoint = api_endpoint

    @property
    def name(self) -> str:
        return "my_custom_router"

    def compute_isochrone(self, request: TravelTimeRequest) -> TravelTimeResult:
        # Replace this with your actual API call
        # Here we just return a fake 1km circle for demonstration
        origin = Point(request.origin_lon, request.origin_lat)
        polygon = origin.buffer(0.01)  # ~1 km at equator

        import geopandas as gpd

        iso_gdf = gpd.GeoDataFrame(
            {
                "geometry": [polygon],
                "duration_min": [request.duration_minutes],
                "mode": [request.mode.value],
                "time_context": [request.time_context.value],
            },
            crs="EPSG:4326",
        )
        return TravelTimeResult(
            request=request,
            isochrone_gdf=iso_gdf,
            network_gdf=None,
            poi_gdf=None,
            router_name=self.name,
        )


# Use it exactly like any other router
custom = MyCustomRouter(api_endpoint="https://my-routing-api.example.com")
result_custom = compute(request, router=custom)
print(f"Router: {result_custom.router_name}")
print(f"Isochrone rows: {len(result_custom.isochrone_gdf)}")

## 6 · Auto-selection logic

`compute()` with no `router=` argument calls `_default_router()`:
1. If `ORS_API_KEY` is set → `OpenRouteServiceRouter`
2. Otherwise → `OsmnxRouter`


In [ ]:
import os

from idd_travel_time.isochrone.compute import _default_router

# Demonstrate auto-selection
if os.environ.get("ORS_API_KEY"):
    print("ORS_API_KEY is set → will use OpenRouteServiceRouter")
else:
    print("No ORS_API_KEY → will use OsmnxRouter")

router = _default_router()
print(f"Selected: {router.name}")

## 7 · Choosing the right backend

| Backend | Key | Speed | Best for |
|---|---|---|---|
| `OsmnxRouter` | None | Slow first run, fast with cache | Development, batch, cluster |
| `OpenRouteServiceRouter` | ORS (free) | Fast | Interactive use, notebooks |
| `GoogleMapsRouter` | Google (paid) | Fast | High-accuracy production use |
| Custom | Yours | Depends | Any travel-time matrix source |

**Rule of thumb:**
- Proof-of-principle on select cities → `OsmnxRouter` + `GraphCache`
- Interactive exploration → `OpenRouteServiceRouter` (set `ORS_API_KEY`)
- Global production run → `OsmnxRouter` + `GraphCache` + `compute_parallel`
